# Is it skill? Choosing a randomization test

A backtest that makes money has three possible sources: an edge in the market,
plain exposure to a trend, or the search that found the strategy. A
randomization test separates them. It reruns the backtest many times on data
where the link between the strategy and the market is broken, and reports the
p-value: the share of those runs that do at least as well as the real one.

screamer has several such tests, and each breaks the link in a different way.
This notebook runs them on two synthetic markets whose answer is known:

- **Market A** is a random walk. No strategy has an edge on it.
- **Market B** is mean-reverting. A z-score reversion rule has a real edge.

The same tests run on both, so you can see which ones get each market right.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import (RollingZscore, BandSignal, BacktestOHLCTarget, position_test,
                      strategy_test, deflated_sharpe)

T, SUB = 2000, 6             # 2000 hourly bars, each built from 6 price steps
PPY = 24 * 365               # bars per year, to annualize the Sharpe ratio
STEP = 0.01 / np.sqrt(SUB)   # 1% volatility per bar


def bars(logp):
    # open, high, low, close of each bar from a log-price path
    p = np.exp(logp).reshape(T, SUB)
    c = p[:, -1]
    o = np.r_[p[0, 0], c[:-1]]
    return o, np.maximum(p.max(axis=1), o), np.minimum(p.min(axis=1), o), c


def random_walk(seed):
    rng = np.random.default_rng(seed)
    return bars(np.log(100) + np.cumsum(rng.standard_normal(T * SUB) * STEP))


def mean_reverting(seed, theta=0.02):
    # log price pulled back to log(100) by 2% of its distance per bar
    rng = np.random.default_rng(seed)
    x = np.zeros(T * SUB)
    e = rng.standard_normal(T * SUB) * STEP
    for i in range(1, T * SUB):
        x[i] = x[i - 1] * (1 - theta / SUB) + e[i]
    return bars(np.log(100) + x)


A = random_walk(7)
B = mean_reverting(8)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(A[3], lw=0.8, label="A: random walk")
ax.plot(B[3], lw=0.8, label="B: mean-reverting")
ax.set_xlabel("bar"); ax.set_ylabel("close"); ax.legend()
plt.tight_layout()

## Two strategies

Both strategies trade a z-score band: short when the log price is far above its
trailing mean, long when it is far below, flat again at the mean
([`BandSignal`](../functions_signal/BandSignal.md) on a
[`RollingZscore`](../functions_rolling/RollingZscore.md)). Each backtest pays a
5 basis point fee per trade.

- The **fixed rule** uses one setting, a 50-bar window and an entry at 1.5,
  chosen in advance.
- The **tuned strategy** tries 30 settings (6 windows and 5 entry levels) on
  the data and keeps the one with the best Sharpe ratio. This is what a
  parameter search does.

In [ ]:
engine = BacktestOHLCTarget(taker_fee=5e-4)
WINDOWS = [10, 20, 30, 50, 80, 120]
ENTRIES = [0.5, 1.0, 1.5, 2.0, 2.5]


def grid(c):
    # one target column per setting, (T, 30)
    cols = [BandSignal(entry=k, exit=0.0)(RollingZscore(w)(np.log(c)))
            for w in WINDOWS for k in ENTRIES]
    return np.nan_to_num(np.column_stack(cols))


def sharpe(out):
    # annualized Sharpe ratio of each backtest in an engine output; NaN for a
    # setting that never trades
    pnl = out[..., 1]
    sd = pnl.std(axis=0, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(sd > 0, pnl.mean(axis=0) / sd, np.nan) * np.sqrt(PPY)


def fixed_rule(o, h, l, c):
    target = np.nan_to_num(BandSignal(entry=1.5, exit=0.0)(RollingZscore(50)(np.log(c))))
    return engine(target, o, h, l, c)


def tuned(o, h, l, c):
    # search the 30 settings on this data and return the best one's backtest
    runs = engine(grid(c), o, h, l, c)
    return runs[:, int(np.nanargmax(sharpe(runs)))]


for name, m in [("A", A), ("B", B)]:
    print(f"market {name}: fixed rule Sharpe {sharpe(fixed_rule(*m)):5.2f}, "
          f"tuned strategy Sharpe {sharpe(tuned(*m)):5.2f}")

On the random walk the fixed rule loses money, as it should, but the tuned
strategy shows a healthy Sharpe ratio: the best of 30 settings looks good by
chance. On the mean-reverting market both make money.

## Market A: an edge that is not there

Four ways to ask whether the tuned strategy's result is luck:

- [`position_test`](../functions_fin/position_test.md) keeps the market and the
  chosen strategy's trades, and shifts the trades to random times. It asks
  whether the timing beats random timing with the same trades.
- `position_test` with all 30 settings as a `(T, 30)` target shifts them
  together and returns `family_p_value`, which compares the chosen setting with
  the best of the 30 in every shifted round.
- [`deflated_sharpe`](../functions_fin/deflated_sharpe.md) needs no
  simulation: it raises the bar to the Sharpe ratio the best of the 30 would
  reach by luck.
- [`strategy_test`](../functions_fin/strategy_test.md) keeps the strategy and
  randomizes the market: it permutes the bars (see
  [`permute_bars`](../functions_fin/permute_bars.md)) and reruns the whole
  strategy, search included, on every copy.

In [ ]:
def four_tests(m, n=999):
    o, h, l, c = m
    targets = grid(c)
    runs = engine(targets, o, h, l, c)
    best = int(np.nanargmax(sharpe(runs)))
    single = position_test(engine, targets[:, best], o, h, l, c, n=n, periods_per_year=PPY)
    family = position_test(engine, targets, o, h, l, c, n=n, periods_per_year=PPY)
    trials = sharpe(runs) / np.sqrt(PPY)             # per-bar Sharpe of every setting
    deflated, _ = deflated_sharpe(runs[:, best], trials[np.isfinite(trials)])
    whole = strategy_test(tuned, o, h, l, c, n=n // 2, periods_per_year=PPY)
    print(f"  position_test, chosen setting  p = {single.p_value:.3f}")
    print(f"  position_test, family          p = {family.family_p_value[best]:.3f}")
    print(f"  deflated_sharpe                P(edge) = {deflated:.3f}")
    print(f"  strategy_test, whole search    p = {whole.p_value:.3f}")
    return single, whole


print("market A, tuned strategy")
A_single, A_whole = four_tests(A)

`position_test` on the chosen setting gives the lowest p-value of the four. It
checks the timing of the final positions, but not how they were found:
the search picked the setting whose timing happened to fit this particular
path, and shifted copies of it do not share that luck. The three tests that
account for the search see nothing.

One market is one draw. Repeating the experiment on 60 independent random walks
shows how often each test is fooled. Where nothing is there, a sound test gives
p below 0.05 in about 5% of the walks, and its p-values spread evenly between 0
and 1.

In [ ]:
p_position, p_strategy = [], []
for k in range(60):
    o, h, l, c = random_walk(100 + k)
    targets = grid(c)
    best = int(np.nanargmax(sharpe(engine(targets, o, h, l, c))))
    p_position.append(position_test(engine, targets[:, best], o, h, l, c, n=199,
                                    periods_per_year=PPY, seed=k).p_value)
    p_strategy.append(strategy_test(tuned, o, h, l, c, n=199,
                                    periods_per_year=PPY, seed=k).p_value)

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, p, title in [(axes[0], p_position, "position_test on the chosen setting"),
                     (axes[1], p_strategy, "strategy_test on the whole search")]:
    ax.hist(p, bins=np.linspace(0, 1, 21), color="lightsteelblue")
    ax.axvline(0.05, color="crimson", lw=1)
    ax.set_title(f"{title}\np < 0.05 in {np.mean(np.array(p) < 0.05):.0%} of walks", fontsize=10)
    ax.set_xlabel("p-value")
axes[0].set_ylabel("random walks")
print(f"p < 0.05: position_test {np.mean(np.array(p_position) < 0.05):.0%}, "
      f"strategy_test {np.mean(np.array(p_strategy) < 0.05):.0%}")
plt.tight_layout()

`position_test` on the chosen setting finds an edge in far more of the walks
than chance allows, and its p-values crowd towards zero. `strategy_test` on the
whole search stays close to 5%. The rule: when the strategy was chosen or
fitted on the data, test the whole procedure with `strategy_test`, or account
for the search with `family_p_value` or `deflated_sharpe`. `position_test`
alone is sound only for a strategy fixed before looking at the data.

## Market B: a real edge

The same four tests on the mean-reverting market.

In [ ]:
print("market B, tuned strategy")
B_single, B_whole = four_tests(B)

Both randomization tests find the edge, the search included. `deflated_sharpe`
is less sure. It treats the spread of the 30 settings' Sharpe ratios as luck,
and on a market with a real edge that spread is large, because some settings
catch the reversion far better than others. That makes it the most
conservative of the four.

## Every statistic at once

A p-value covers one statistic, the Sharpe ratio here. Each test also returns
every [`backtest_report`](../functions_fin/backtest_report.md) statistic of the
real run (`observed_stats`) and of each randomized run (`null_stats`), so the
real run can be placed against the randomized ones on two statistics at once.
Each dot below is one randomized run and the star is the real run, for the
tuned strategy on both markets.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
panels = [(A_single, "A: position_test"), (A_whole, "A: strategy_test"),
          (B_single, "B: position_test"), (B_whole, "B: strategy_test")]
for ax, (res, title) in zip(axes.flat, panels):
    null, real = res.null_stats, res.observed_stats
    ax.scatter(null["sharpe"], null["max_drawdown"], s=6, color="lightsteelblue",
               label="randomized runs")
    ax.scatter(real["sharpe"], real["max_drawdown"], s=250, marker="*", color="crimson",
               label=f"real run, p = {res.p_value:.3f}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Sharpe ratio"); ax.set_ylabel("max drawdown ($)")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()

On market B the star lies beyond the `position_test` cloud, with a higher
Sharpe ratio and a smaller drawdown than any randomized run, and at the edge of
the `strategy_test` cloud. On market A the star sits in the upper part of the
`position_test` cloud, which holds random timings of trades that were chosen to
fit this path. It sits inside the `strategy_test` cloud, which holds the
results of the same search on permuted markets: many of them give the search a
higher Sharpe ratio than the real path does.

## Shift or segments

`position_test` has two ways to move the trades in time:

- `method="shift"` (the default) rotates the whole target series by a random
  offset. Every property of the positions is kept, including how the trades
  cluster in time.
- `method="segments"` shuffles the order of the runs: each trade and each flat
  stretch. The same trades occur, but their clustering is broken.

Use `"shift"` by default. Use `"segments"` when the null should not keep the
order of the trades, for example to check that the result does not come from a
few trades that happen to fall in one favourable stretch. On the fixed rule in
market B both agree.

In [ ]:
o, h, l, c = B
target = np.nan_to_num(BandSignal(entry=1.5, exit=0.0)(RollingZscore(50)(np.log(c))))

fig, ax = plt.subplots(figsize=(9, 3))
for method, color in [("shift", "lightsteelblue"), ("segments", "burlywood")]:
    res = position_test(engine, target, o, h, l, c, n=999, method=method, periods_per_year=PPY)
    ax.hist(res.null, bins=40, color=color, alpha=0.7, label=f"{method}, p = {res.p_value:.3f}")
ax.axvline(res.observed, color="crimson", lw=2, label="real run")
ax.set_xlabel("Sharpe ratio"); ax.legend()
plt.tight_layout()

## Which inputs carry the edge?

`strategy_test` with `method="shift"` keeps the prices and rotates chosen
inputs, the features, by one random offset. It asks whether those features
carry information about the market. Here the fixed rule's band runs on a
feature passed as a fifth input: the price z-score itself, or a smoothed noise
series with the same scale.

In [ ]:
def band_on(o, h, l, c, feature):
    return engine(np.nan_to_num(BandSignal(entry=1.5, exit=0.0)(feature)), o, h, l, c)


zscore = RollingZscore(50)(np.log(c))
rng = np.random.default_rng(3)
noise = RollingZscore(50)(np.convolve(rng.standard_normal(T), np.ones(50) / 50, mode="same"))

for name, feature in [("price z-score", zscore), ("noise", noise)]:
    res = strategy_test(band_on, o, h, l, c, feature, n=499, method="shift", columns=(4,),
                        periods_per_year=PPY)
    print(f"{name:14s} Sharpe {res.observed:6.2f}, p = {res.p_value:.3f}")

The z-score carries the edge and the noise does not. With several features,
list them all in `columns`: they move together, so their relations to each
other are kept and only their alignment with the market is broken.

## Which test when

| Question | Test | Randomizes | Keeps | Reruns the strategy |
|---|---|---|---|---|
| Does the timing beat random timing with the same trades? | `position_test`, `method="shift"` | when the trades happen | every trade, exposure, clustering | no |
| The same, without the order of the trades | `position_test`, `method="segments"` | the order of the trades | every trade and its length | no |
| Does the best of several variants survive the choice? | `position_test` with a `(T, K)` target, `family_p_value` | all variants together | relations between variants | no |
| Does a portfolio's timing beat random timing? | `position_test` with a portfolio engine | the whole book together | relations between assets | no |
| Is there market structure the whole strategy, fitting included, uses? | `strategy_test`, `method="bars"` | the order of the bars | distribution of bars, first and last price | yes |
| The same, for a strategy on one price series | `strategy_test`, `method="returns"` | the order of the returns | distribution of returns | yes |
| Do these features carry information? | `strategy_test`, `method="shift"`, `columns` | alignment of the features | each feature's own structure | yes |
| Is the Sharpe ratio above zero, or above the best of the variants tried? | `probabilistic_sharpe`, `deflated_sharpe` | nothing (analytic) | | no |

Points to watch:

- Exposure is not skill. A long-biased strategy in a rising market makes
  money; `position_test` keeps its exposure in every randomized run, so it
  tests only the timing.
- Permuting bars with `mode="log"` needs positive prices. Use `"diff"` for
  spreads or rates that can be zero or negative, and `"level"` for series that
  are stationary in level.
- With `method="bars"` or `"returns"`, every other input moves with its bar.
  List inputs that must stay in place, such as timestamps, in `fixed`. A volume
  input moves unchanged (`volume_mode="keep"`) or is rescaled so the traded
  amount stays that of its source bar (`"notional"`).
- `position_test` does not support the order-posting engines (`*Orders`),
  whose limit prices are tied to the market at each moment.

The [significance recipe](../recipes/significance-test.md) runs `position_test`
on a real BTC tape, and the [backtesting guide](../backtesting_best_practices.md)
places these tests among the other checks of a backtest.